# Dataset visualization

This notebook provides tools for visualizing the global snowmelt runoff onset dataset.

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import contextily as ctx
import matplotlib.pyplot as plt
import xyzservices as xyz
import coiled
#import hvplot.xarray
#import cartopy.crs as ccrs
from global_snowmelt_runoff_onset.config import Config, Tile
import easysnowdata

In [2]:
config = Config('config/global_config_v9.txt')

SAS token is valid until 2026-02-26 21:38 UTC (552.3 hours)
----------------------------------------
Configuration loaded:
config_name = global_config_v9
version = v9
resolution = 0.00072000072000072
bands = vv
mountain_snow_only = False
spatial_chunk_dim_s1_read = 2048
spatial_chunk_dim_s1_process = 512
spatial_chunk_dim_zarr_output = 2048
bbox_left = -179.999
bbox_right = 179.999
bbox_top = 81.099
bbox_bottom = -59.999
wy_start = 2015
wy_end = 2024
low_backscatter_threshold = 0.001
min_monthly_acquisitions = 1
max_allowed_days_gap_per_orbit = 30
min_years_for_median_std = 3
extend_search_window_beyond_sdd_days = 16
min_consec_snow_days_for_seasonal_snow = 56
valid_tiles_geojson_path = processing/tile_data/global_tiles_with_seasonal_snow.geojson
tile_results_path = processing/tile_data/tile_results_v9.csv
global_runoff_zarr_store_azure_path = snowmelt/snowmelt_runoff_onset/global_v9.zarr
seasonal_snow_mask_zarr_store_azure_path = snowmelt/snow_cover/global_modis_snow_cover.zarr
season

In [ ]:
s1_dims_from_tiles = config.valid_tiles_gdf['s1_rtc_ds_dims']
s1_dims_from_tiles

In [ ]:
import ast

total_time_steps = sum([ast.literal_eval(dims_str)['time'] for dims_str in s1_dims_from_tiles if pd.notna(dims_str)])
total_time_steps

In [ ]:
m=config.valid_tiles_gdf.explore(column='success',cmap=['red','green'], tiles=xyz.providers.Esri.WorldImagery)
tiles_with_no_vv_data = config.valid_tiles_gdf[(config.valid_tiles_gdf['error_messages'].str.contains('No such band/alias',na=False) | config.valid_tiles_gdf['error_messages'].str.contains('empty sequence',na=False))]
tiles_with_no_vv_data.explore(m=m,column='success',cmap=['orange'])

In [3]:
global_ds = xr.open_zarr(config.global_runoff_store, consolidated=True, decode_coords='all')

def view_tile(tile: Tile):


    test_ds = global_ds.rio.clip_box(*tile.get_geobox().boundingbox,crs='EPSG:4326')
    test_ds = test_ds.rio.reproject(test_ds.rio.estimate_utm_crs())

    f,axs=plt.subplots(1,3,figsize=(15,5))
    test_ds['runoff_onset_median'].plot(ax=axs[0],vmin=0,vmax=365)
    axs[0].set_title('2015-2024 median snowmelt runoff onset')

    if 'runoff_onset_std' in test_ds:
        test_ds['runoff_onset_std'].plot(ax=axs[1],cmap='Reds',vmin=0,vmax=60)
        axs[1].set_title('2015-2024 std deviation snowmelt runoff onset')
    elif 'runoff_onset_mad' in test_ds:
        test_ds['runoff_onset_mad'].plot(ax=axs[1],cmap='Reds',vmin=0,vmax=60)
        axs[1].set_title('2015-2024 median absolute deviation snowmelt runoff onset')
    
    if 'temporal_resolution_median' in test_ds:
        test_ds['temporal_resolution_median'].plot(ax=axs[2],cmap='summer',vmin=0,vmax=30)
        axs[2].set_title('Median temporal resolution (days)')

    for ax in axs:
        ctx.add_basemap(ax=ax, crs=test_ds.rio.crs.to_string())
        ax.set_aspect('equal')
        

    f.tight_layout()

    test_ds['runoff_onset'].plot.imshow(col='water_year',col_wrap=5,vmin=0,vmax=365)

    (test_ds['runoff_onset']-test_ds['runoff_onset_median']).plot.imshow(col='water_year',col_wrap=5,vmin=-60,vmax=60,cmap='RdBu')

    test_ds['temporal_resolution'].plot.imshow(col='water_year',col_wrap=5,vmin=0,vmax=30,cmap='summer')

In [ ]:
# global_ds = xr.open_zarr(config.global_runoff_store, consolidated=True, decode_coords='all')
# global_ds

# csg_gdf = gpd.read_file("geometries/utah_and_hma/CSG_Himalaya.shp")
# csg_gdf

# dd_gdf = gpd.read_file("geometries/utah_and_hma/DD_Himalaya.shp")
# dd_gdf

# f,ax=plt.subplots(1,1,figsize=(8,8))
# csg_gdf.plot(ax=ax,facecolor='none',edgecolor='blue',linewidth=2)
# dd_gdf.plot(ax=ax,facecolor='none',edgecolor='red',linewidth=2)
# ctx.add_basemap(ax=ax, crs='EPSG:4326', source=xyz.providers.Esri.WorldImagery)


# utah_gdf = gpd.read_file("geometries/utah_and_hma/Utah_site.shp")
# utah_gdf

# csg_ds = global_ds.rio.clip_box(*csg_gdf.total_bounds,crs='EPSG:4326').compute()
# csg_ds

# csg_ds.to_netcdf("geometries/utah_and_hma/csg_himalaya_runoff_onset.nc")

# dd_ds = global_ds.rio.clip_box(*dd_gdf.total_bounds,crs='EPSG:4326').compute()
# dd_ds.to_netcdf("geometries/utah_and_hma/dd_himalaya_runoff_onset.nc")

# utah_ds = global_ds.rio.clip_box(*utah_gdf.total_bounds,crs='EPSG:4326').compute()
# utah_ds.to_netcdf("geometries/utah_and_hma/utah_site_runoff_onset.nc")

# test_dd_ds = xr.open_dataset("geometries/utah_and_hma/dd_himalaya_runoff_onset.nc", decode_coords='all')
# test_dd_ds

# test_dd_ds['runoff_onset'].plot.imshow(col='water_year',col_wrap=5,cmap='viridis',robust=True, cbar_kwargs={'label':'day of water year'})

# test_ds['temporal_resolution'].plot.imshow(col='water_year',col_wrap=5,cmap='YlGn_r', robust=True)

# f,ax=plt.subplots(1,3,figsize=(20,5))
# test_dd_ds['runoff_onset_median'].plot(ax=ax[0],vmin=220,vmax=310,cbar_kwargs={'label':'day of water year'})
# ax[0].set_title('2015-2024 median snowmelt runoff onset')
# test_dd_ds['runoff_onset_mad'].plot(ax=ax[1],cmap='Reds',vmin=0,vmax=30,cbar_kwargs={'label':'days'})
# ax[1].set_title('2015-2024 snowmelt runoff onset median absolute deviation')
# test_dd_ds['temporal_resolution_median'].plot(ax=ax[2],cmap='YlGn_r',cbar_kwargs={'label':'days'})
# ax[2].set_title('2015-2024 median temporal resolution')


In [11]:
# test_dd_ds['runoff_onset'].plot.imshow(col='water_year',col_wrap=5,cmap='viridis',robust=True, cbar_kwargs={'label':'day of water year'})


In [12]:
# snotels = easysnowdata.automatic_weather_stations.StationCollection()
# snotels

In [13]:
# snotels.all_stations

In [14]:
# snotels.get_entire_data_archive()

In [15]:
# snotel_ds = snotels.entire_data_archive
# snotel_ds

In [16]:
# f,ax=plt.subplots(1,1,figsize=(20,10))
# snotel_ds['WTEQ'].sel(station='628_UT_SNTL').plot(ax=ax)
# ax.set_xlim(pd.to_datetime('2014-10-01'),pd.to_datetime('2024-09-30'))
# # plot red dashed line last day for each water year when SWE goes to 95% of max for that water year and annotate with date
# for wy in range(2015,2025):
#     wy_start = pd.to_datetime(f'{wy-1}-10-01')
#     wy_end = pd.to_datetime(f'{wy}-09-30')
#     wy_data = snotel_ds['WTEQ'].sel(station='628_UT_SNTL').sel(time=slice(wy_start, wy_end))
#     max_swe = wy_data.max().item()
#     threshold = 0.95 * max_swe
#     days_above_threshold = wy_data.where(wy_data >= threshold, drop=True)
#     if len(days_above_threshold.time) > 0:
#         last_day = days_above_threshold.time.values[-1]
#         ax.axvline(last_day, color='red', linestyle='--')
#         ax.annotate(f'{pd.to_datetime(last_day).strftime("%Y-%m-%d")}', xy=(last_day, threshold), xytext=(last_day, threshold),
#                     arrowprops=dict(facecolor='black', arrowstyle='->'),
#                     horizontalalignment='center')
        
# for wy in range(2016,2024):
#     wy_start = pd.to_datetime(f'{wy-1}-10-01')
#     wy_end = pd.to_datetime(f'{wy}-09-30')
#     swe_wy = snotel_ds['WTEQ'].sel(station='628_UT_SNTL').sel(time=slice(wy_start,wy_end))
#     max_swe = swe_wy.max().item()
#     threshold = 0.95 * max_swe
#     days_above_threshold = swe_wy.where(swe_wy >= threshold, drop=True)
#     if days_above_threshold.time.size > 0:
#         last_day = days_above_threshold.time.values[-1]
#         ax.axvline(last_day, color='red', linestyle='--')

In [17]:
# f,ax=plt.subplots(1,1,figsize=(10,10))
# utah_gdf.plot(ax=ax,facecolor='none',edgecolor='blue',linewidth=2)
# ctx.add_basemap(ax=ax, crs='EPSG:4326', source=xyz.providers.Esri.WorldImagery)
# snotels.all_stations.plot(ax=ax,marker='^',color='yellow',markersize=50, label='SNOTEL Stations')
# ax.set_xlim( -111.8, -111.5)
# ax.set_ylim( 39.2, 39.5)

In [ ]:
# config.azure_blob_fs.download('snowmelt/snowmelt_runoff_onset/global_v5.zarr',lpath="/mnt/c/Users/elgag/Downloads/global_v5.zarr",recursive=True)
# config.azure_blob_fs.ls('snowmelt/snowmelt_runoff_onset/global_v5.zarr',detail=True)

In [ ]:
url = (f"https://data.earthenv.org/mountains/standard/GMBA_Inventory_v2.0_standard_300.zip")
gmba_gdf = gpd.read_file("zip+" + url)
chersky_range_gdf = gmba_gdf[gmba_gdf['MapName'] == 'Chersky Range']
chersky_range_gdf

In [ ]:
chersky_range_gdf.plot()

In [ ]:
store = config.azure_blob_fs.get_mapper(f"snowmelt/snowmelt_runoff_onset/coarsened/global_v9_coarsened_20_ds.zarr")
runoff_onset_ds = xr.open_zarr(store, consolidated=True, decode_coords='all', chunks="auto").rio.write_crs("EPSG:4326")
runoff_onset_ds

In [ ]:
chersky_range_ds = runoff_onset_ds.rio.clip_box(*chersky_range_gdf.total_bounds,crs='EPSG:4326')
chersky_range_ds

In [ ]:
chersky_range_ds['runoff_onset_anomaly'] = chersky_range_ds['runoff_onset'] - chersky_range_ds['runoff_onset_median']
chersky_range_ds

In [ ]:
chersky_range_ds['runoff_onset_anomaly'].plot.imshow(col='water_year',col_wrap=5,vmin=-40,vmax=40,cmap='RdBu')

In [ ]:
chersky_range_ds['runoff_onset_anomaly'].where(chersky_range_ds['temporal_resolution']<14).plot.imshow(col='water_year',col_wrap=5,vmin=-40,vmax=40,cmap='RdBu')

In [ ]:
chersky_range_ds['temporal_resolution'].plot.imshow(col='water_year',col_wrap=5)

In [ ]:
#view_tile(config.get_tile(26,128))
#view_tile(config.get_tile(19,33))
#view_tile(config.get_tile(26,63))
#view_tile(config.get_tile(15,120))
view_tile(config.get_tile(21,128))


In [ ]:
view_tile(config.get_tile(10,109))

In [ ]:
view_tile(config.get_tile(23,39))

In [ ]:
view_tile(config.get_tile(10,109))

In [ ]:
view_tile(config.get_tile(16,139))

In [ ]:
view_tile(config.get_tile(23,127))

In [ ]:
view_tile(tile)

In [ ]:
test_ds['runoff_onset'].sel(water_year=2020).

In [ ]:
# print percentiles for each water year
for WY in test_ds['water_year'].values:
    print(f"Water Year {WY}:")
    print(f"runoff_onset percentiles: {test_ds['runoff_onset'].sel(water_year=WY).quantile([0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]).values}")
    print(f"runoff_onset number of nans: {test_ds['runoff_onset'].sel(water_year=WY).isnull().sum().values}")
    print(f"temporal_resolution percentiles: {test_ds['temporal_resolution'].sel(water_year=WY).quantile([0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]).values.round(decimals=1)}")
    print(f"temporal_resolution number of nans: {test_ds['temporal_resolution'].sel(water_year=WY).isnull().sum().values}")
    print()
    

In [ ]:
for WY in test_ds_2['water_year'].values:
    print(f"Water Year {WY}:")
    print(f"runoff_onset percentiles: {test_ds_2['runoff_onset'].sel(water_year=WY).quantile([0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]).values}")
    print(f"runoff_onset number of nans: {test_ds_2['runoff_onset'].sel(water_year=WY).isnull().sum().values}")
    print(f"temporal_resolution percentiles: {test_ds_2['temporal_resolution'].sel(water_year=WY).quantile([0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]).values.round(decimals=1)}")
    print(f"temporal_resolution number of nans: {test_ds_2['temporal_resolution'].sel(water_year=WY).isnull().sum().values}")
    print()

In [ ]:
#tile = config.get_tile(23,39)
tile = config.get_tile(23,39)
tile = config.get_tile(12,203)
tile = config.get_tile(13,156)


In [ ]:
tile.geobox.explore()

In [ ]:
view_tile(tile)

In [ ]:
view_tile(config.get_tile(23,39))

In [ ]:
coords = (-135, 58, -133, 66)
bbox_gdf = easysnowdata.utils.convert_bbox_to_geodataframe(bbox_input=coords)
bbox_gdf

In [ ]:
bbox_gdf.explore()

In [ ]:
test_ds = global_ds.rio.clip_box(*bbox_gdf.total_bounds,crs=bbox_gdf.crs)
test_ds

In [ ]:
f,ax=plt.subplots()
test_ds['runoff_onset_mad'].plot.imshow(ax=ax,cmap='Reds',vmin=0,vmax=30)#.sel(latitude=slice(70,60),longitude=slice(-170,-50))
ax.set_aspect('equal')

In [ ]:
(test_ds['runoff_onset']-test_ds['runoff_onset_median']).plot.imshow(col='water_year',col_wrap=5,vmin=-60,vmax=60,cmap='RdBu')

In [ ]:
# global_ds = xr.open_zarr(config.global_runoff_store,mask_and_scale=False,consolidated=True, decode_coords='all').sel(longitude=slice(-125,-105),latitude=slice(50,35))#colorado #.sel(longitude=slice(-109,-105),latitude=slice(41,37)) # WUS: .sel(longitude=slice(-125,-105),latitude=slice(50,35))
# global_ds


# nodata_uint16 = 0
# nodata_float32 = np.finfo(np.float32).min
# chunk_size = (256, 256)
# import zarr

# encoding = {
#         "water_year": {
#             "chunks": global_ds.water_year.shape,
#             "compressor": zarr.Blosc(cname="zstd"),
#         },
#         "runoff_onset": {
#             "chunks": (1,) + chunk_size,
#             "compressor": zarr.Blosc(cname="zstd"),
#             "dtype": "uint16",
#         },
#         "runoff_onset_median": {
#             "chunks": chunk_size,
#             "compressor": zarr.Blosc(cname="zstd"),
#             "dtype": "uint16",
#         },
#         "runoff_onset_mad": {
#             "chunks": chunk_size,
#             "compressor": zarr.Blosc(cname="zstd"),
#             "dtype": "float32",
#         },
#         "latitude": {
#             "chunks": chunk_size[0],
#             "compressor": zarr.Blosc(cname="zstd"),
#         },
#         "longitude": {
#             "chunks": chunk_size[1],
#             "compressor": zarr.Blosc(cname="zstd"),
#         },
#         }

# global_ds.chunk(latitude=256,longitude=256).to_zarr('/mnt/c/Users/elgag/Downloads/WUS_runoff_onset.zarr', mode='w', consolidated=True, write_empty_chunks=False, encoding=encoding)

# colorado_test_ds = xr.open_zarr('/mnt/c/Users/elgag/Downloads/colorado_runoff_onset.zarr', consolidated=True, decode_coords='all')
# colorado_test_ds
# colorado_test_ds['runoff_onset_median'].plot.imshow()

# #colorado_test_ds['runoff_onset'].plot.imshow(col='water_year',col_wrap=5,vmin=0,vmax=365)
# WUS_ds = xr.open_zarr('/mnt/c/Users/elgag/Downloads/WUS_runoff_onset.zarr', consolidated=True, decode_coords='all')
# WUS_ds
# colorado_test_ds['runoff_onset_median'].plot.imshow()

#colorado_test_ds['runoff_onset'].plot.imshow(col='water_year',col_wrap=5,vmin=0,vmax=365)

In [ ]:
tile = config.get_tile(27,50)

In [ ]:
tile = config.get_tile(62,71)
tile = config.get_tile(62,72)
tile = config.get_tile(63,71)

In [ ]:
tile = config.get_tile(34,185)
tile = config.get_tile(31,174)
tile = config.get_tile(31,176)
tile = config.get_tile(90,72)
tile = config.get_tile(91,72)

In [ ]:
tile = config.get_tile(13,26)
tile = config.get_tile(13,25)

In [ ]:
view_tile(tile)

In [ ]:
test_ds = global_ds.rio.clip_box(*tile.get_geobox().boundingbox,crs='EPSG:4326')
test_ds = test_ds.rio.reproject(test_ds.rio.estimate_utm_crs())
test_ds

In [ ]:
f,axs=plt.subplots(1,2,figsize=(30,15))
test_ds['runoff_onset_median'].plot.imshow(ax=axs[0])
test_ds['runoff_onset_median'].plot(ax=axs[1])
for ax in axs:
    ctx.add_basemap(ax=ax, crs=test_ds.rio.crs, source=ctx.providers.Esri.WorldImagery)
    ax.set_aspect('equal')

In [ ]:
# config.azure_blob_fs.download('snowmelt/snowmelt_runoff_onset/global_v5.zarr',lpath="/mnt/c/Users/elgag/Downloads/global_v5.zarr",recursive=True)
# config.azure_blob_fs.ls('snowmelt/snowmelt_runoff_onset/global_v5.zarr',detail=True)

In [ ]:
tile = config.get_tile(64,75)

In [ ]:
seasonal_snow_mask = xr.open_zarr(config.snow_phenology_store, consolidated=True, decode_coords='all') 
seasonal_snow_mask_clip_ds = seasonal_snow_mask.rio.clip_box(*tile.bbox_gdf.total_bounds,crs='EPSG:4326')
seasonal_snow_mask_clip_ds

In [ ]:
seasonal_snow_mask_clip_ds['max_consec_snow_days'].plot.imshow(col='water_year',col_wrap=5,vmin=0,vmax=100)

In [ ]:
tile = config.get_tile(63,70)
view_tile(tile) # v1

In [ ]:
tile = config.get_tile(63,71)
view_tile(tile) # v1

In [ ]:
tile = config.get_tile(31,194)
view_tile(tile) # v1

In [ ]:
test_gdf = gmba_gdf[gmba_gdf['MapName']=='Canadian Rockies']
test_gdf = gmba_gdf[gmba_gdf['MapName']=='Qiangtang']
test_gdf = gmba_gdf[gmba_gdf['MapName']=='Yukon Intermountain Ranges']
test_gdf = gmba_gdf[gmba_gdf['MapName']=='Qin Ling']
test_gdf = gmba_gdf[gmba_gdf['GMBA_V2_ID']==15623]


test_gdf

In [ ]:
#test_gdf.to_file('cordillera_central_central_andes.geojson')

In [ ]:
test_gdf.explore()

In [ ]:
#test_gdf.explore(tiles=xyz.providers.Esri.WorldImagery)
8.5,9.75,-78, -76.3

In [ ]:
#test_ds = global_ds['runoff_onset_median'].rio.clip_box(*test_gdf.total_bounds,crs='EPSG:4326').coarsen(latitude=20,longitude=20,boundary='trim').mean()
#test_ds = global_ds.rio.clip_box(-78,-9.75,-76.3,-8.5,crs='EPSG:4326')
#test_ds = global_ds.rio.clip_box(-75.5,-12,-74.5,-11,crs='EPSG:4326')
test_ds = global_ds['runoff_onset_median'].rio.clip_box(-79,-13,-74,-11,crs='EPSG:4326')


test_ds

In [ ]:
fcf_da = easysnowdata.remote_sensing.get_forest_cover_fraction((-75.5,-12,-74.5,-11),mask_nodata=True)
fcf_da

In [ ]:
worldcover_da = easysnowdata.remote_sensing.get_esa_worldcover((-75.5,-12,-74.5,-11),mask_nodata=True)
worldcover_da

In [ ]:
fcf_da.plot.imshow()

In [ ]:
import rasterio as rio
worldcover_da = worldcover_da.rio.reproject_match(fcf_da,resampling=rio.enums.Resampling.mode)
worldcover_da

In [ ]:
f,ax = worldcover_da.attrs['example_plot'](worldcover_da)

In [ ]:
xr.where(((worldcover_da==10) & (fcf_da==0)),1,0).plot.imshow()

In [ ]:
test_ds = test_ds.rio.reproject(test_ds.rio.estimate_utm_crs())
test_ds

In [ ]:
test_ds.odc.explore(tiles=xyz.providers.Esri.WorldImagery,opacity=0.5)

In [ ]:
test_ds['runoff_onset'].count(dim='water_year').odc.explore(tiles=xyz.providers.Esri.WorldImagery,opacity=0.5)

In [ ]:
filepath_NR = 'BPR_Shapefiles/NavajoRiver_Basin_toBPRGauge.shp'
filepath_NR_to_O = 'BPR_Shapefiles/NavajoRiver_Basin_toOsoDiv.shp'
filepath_BPR = 'BPR_Shapefiles/BPR_Parcels.shp'


In [ ]:
bpr_gdf = gpd.read_file(filepath_BPR).to_crs('EPSG:4326')
nr_gdf = gpd.read_file(filepath_NR).to_crs('EPSG:4326')
nr_to_o_gdf = gpd.read_file(filepath_NR_to_O).to_crs('EPSG:4326')
bpr_gdf

In [ ]:
f,ax=plt.subplots()
nr_to_o_gdf.plot(ax=ax,color='green')
nr_gdf.plot(ax=ax,color='blue')
bpr_gdf.plot(ax=ax,color='red')


In [ ]:
test_ds = global_ds.rio.clip_box(*nr_to_o_gdf.total_bounds,crs='EPSG:4326')
test_ds = test_ds.rio.clip(nr_to_o_gdf.geometry).rio.reproject(test_ds.rio.estimate_utm_crs())
test_ds['runoff_onset_anomaly'] = (test_ds['runoff_onset']-test_ds['runoff_onset_median'])
test_ds = test_ds.compute()
test_ds

In [ ]:
test_gdf = nr_to_o_gdf.to_crs(test_ds.rio.crs.to_string())

In [ ]:
#easysnowdata.utils.datetime_to_DOWY('2022-04-09')
#test_ds['runoff_onset'].where(lambda x: x>191).sel(water_year=2022).plot.imshow()

In [ ]:
f,axs=plt.subplots(1,2,figsize=(10,5))
test_ds['runoff_onset_median'].plot(ax=axs[0],vmin=0,vmax=365)
test_gdf.to_crs(test_ds.rio.crs.to_string()).boundary.plot(ax=axs[0],color='black')
axs[0].set_title('2015-2024 median snowmelt runoff onset')

if 'runoff_onset_std' in test_ds:
    test_ds['runoff_onset_std'].plot(ax=axs[1],cmap='Reds',vmin=0,vmax=60)
    axs[1].set_title('2015-2024 std deviation snowmelt runoff onset')
elif 'runoff_onset_mad' in test_ds:
    test_ds['runoff_onset_mad'].plot(ax=axs[1],cmap='Reds',vmin=0,vmax=60)
    axs[1].set_title('2015-2024 median absolute deviation snowmelt runoff onset')

for ax in axs:
    ctx.add_basemap(ax=ax, crs=test_ds.rio.crs.to_string())
    ax.set_aspect('equal')
    

f.tight_layout()

test_ds['runoff_onset'].plot.imshow(col='water_year',col_wrap=5,vmin=0,vmax=365)

(test_ds['runoff_onset']-test_ds['runoff_onset_median']).plot.imshow(col='water_year',col_wrap=5,vmin=-60,vmax=60,cmap='RdBu')

In [ ]:
water_years = [2017,2018,2019,2020,2021,2022,2023,2024]

f,axs=plt.subplots(2,8,figsize=(15,7),sharex=True,sharey=True,layout='constrained')


for i,water_year in enumerate(water_years):
    test_ds['runoff_onset'].sel(water_year=water_year).plot.imshow(ax=axs[0,i],vmin=100,vmax=280,add_colorbar=False)
    test_ds['runoff_onset_anomaly'].sel(water_year=water_year).plot.imshow(ax=axs[1,i],vmin=-60,vmax=60,cmap='RdBu',add_colorbar=False) #.plot.imshow(col='water_year',col_wrap=5,vmin=-60,vmax=60,cmap='RdBu')

    axs[1,i].set_title('')
    axs[0,i].set_title(f'WY{water_year}')

for ax in axs.flatten():
    ax.axis('off')
    test_gdf.boundary.plot(ax=ax,color='black')
    #ctx.add_basemap(ax=ax, crs=test_ds.rio.crs.to_string())
    ax.set_aspect('equal')

In [ ]:
water_years = [2017,2018,2019,2020,2021,2022,2023,2024]

f,axs=plt.subplots(2,8,figsize=(15,7),sharex=True,sharey=True,layout='constrained')


for i,water_year in enumerate(water_years):
    onset_plot = test_ds['runoff_onset'].sel(water_year=water_year).plot.imshow(ax=axs[0,i],vmin=100,vmax=280,add_colorbar=False)
    anomaly_plot = test_ds['runoff_onset_anomaly'].sel(water_year=water_year).plot.imshow(ax=axs[1,i],vmin=-60,vmax=60,cmap='RdBu',add_colorbar=False) #.plot.imshow(col='water_year',col_wrap=5,vmin=-60,vmax=60,cmap='RdBu')

    axs[1,i].set_title('')
    axs[0,i].set_title(f'WY{water_year}')

for ax in axs.flatten():
    ax.axis('off')
    test_gdf.boundary.plot(ax=ax,color='black')
    #ctx.add_basemap(ax=ax, crs=test_ds.rio.crs.to_string())
    ax.set_aspect('equal')

f.colorbar(onset_plot,ax=axs[0,:],orientation='vertical',label='Runoff onset timing [DOWY]',pad=0.015)
f.colorbar(anomaly_plot,ax=axs[1,:],orientation='vertical',label='Runoff onset timing anomaly [days]',pad=0.015)
f.dpi=300

In [ ]:
# seasonal_snow_mask = xr.open_zarr(config.snow_phenology_store, consolidated=True, decode_coords='all') 
# seasonal_snow_mask_clip_ds = seasonal_snow_mask.rio.clip_box(*tile.bbox_gdf.total_bounds,crs='EPSG:4326')
# seasonal_snow_mask_clip_ds
# seasonal_snow_mask_clip_ds = seasonal_snow_mask_clip_ds.rio.reproject(seasonal_snow_mask_clip_ds.rio.estimate_utm_crs()).rio.clip_box(*tile.bbox_gdf.total_bounds,crs='EPSG:4326')
# seasonal_snow_mask_clip_ds['SAD_DOWY'].where(lambda x: x>0).plot.imshow(col='water_year',col_wrap=5,vmin=0,vmax=365)
# seasonal_snow_mask_clip_ds['SDD_DOWY'].where(lambda x: x>0).plot.imshow(col='water_year',col_wrap=5,vmin=0,vmax=365)
# seasonal_snow_mask_clip_ds['max_consec_snow_days'].where(lambda x: x>0).plot.imshow(col='water_year',col_wrap=5, vmin=0,vmax=365)
# seasonal_snow_mask_clip_ds['SAD_DOWY'].plot.imshow(col='water_year',col_wrap=5)

In [ ]:
cluster = coiled.Cluster(idle_timeout="10 minutes",
                         #shutdown_on_close=False,
                         #wait_for_workers=True,
                         #n_workers=[41,170], # 170
                         #n_workers=[31,86],
                         n_workers=20,
                         #n_workers=8,
                         #n_workers=10,
                         worker_memory="32 GB", #coiled.list_instance_types(backend="azure")
                         #worker_options={"nthreads": 1},
                         #worker_options={"nthreads": 32},# 16 8 4 oversubscribe?
                         #scheduler_memory="128 GB",
                         scheduler_memory="128 GB",
                         spot_policy="spot", # spot usually
                         #software="sar_snowmelt_timing",
                         environ={"GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR"},
                         #container="mcr.microsoft.com/planetary-computer/python:latest",
                         workspace="azure",
                         
                         )

client = cluster.get_client()

In [ ]:
client.restart()

In [ ]:
global_ds = xr.open_zarr(config.global_runoff_store, consolidated=True, decode_coords='all')
global_ds

In [ ]:
coarsen_factor = 100
# coarsened_ds = global_ds.coarsen(latitude=coarsen_factor,longitude=coarsen_factor,boundary='trim').median().compute()
# coarsened_ds

In [ ]:
import dask
@dask.delayed
def coarsen_with_coiled(coarsen_factor):
    global_ds = xr.open_zarr(config.global_runoff_store, consolidated=True, decode_coords='all')
    return global_ds.coarsen(latitude=coarsen_factor,longitude=coarsen_factor,boundary='trim').mean().compute()

coarsen_ds = coarsen_with_coiled(global_ds,coarsen_factor)

In [ ]:
coarsen_ds.compute()

In [ ]:
coarsen_ds = coarsen_ds.persist()
coarsen_ds

In [ ]:
coarsened_computed_ds = coarsen_ds.compute()
coarsened_computed_ds

In [ ]:
coarsened_ds

In [ ]:
client.restart()

In [ ]:
global_ds.persist()

In [ ]:
# global_coarsened_100_ds = global_ds.coarsen(latitude=100,longitude=100,boundary='trim').median().compute()
# global_coarsened_100_ds
# global_coarsened_100_ds.to_netcdf('aggregated_results/global_maps/global_coarsened_100_ds.nc')

In [ ]:
output_path = 'aggregated_results/global_maps/global_coarsened_100_ds.nc'

def coarsen_with_coiled(config, coarsen_factor):
    global_ds = xr.open_zarr(config.global_runoff_store, consolidated=True, decode_coords='all')

    coarsened_ds = global_ds.coarsen(latitude=coarsen_factor,longitude=coarsen_factor,boundary='trim').median()
    coarsened_computed_ds = coarsened_ds.compute()

    return coarsened_computed_ds

future = client.submit(coarsen_with_coiled,config,100)

In [ ]:
coarsened_ds = future.result()
coarsened_ds


In [ ]:
coarsened_ds.to_netcdf(output_path)

In [ ]:
# global_ds['runoff_onset_median'].sel(longitude=slice(-160,-105),latitude=slice(70,30))

In [ ]:
# amco_coarsened_10_da = global_ds['runoff_onset_median'].sel(longitude=slice(-170,-105),latitude=slice(70,30)).coarsen(latitude=10,longitude=10,boundary='trim').median().compute()
# amco_coarsened_10_da
# amco_coarsened_10_da.to_netcdf('amco_N_coarsened_10_da.nc')

# # amco_coarsened_10_da = global_ds['runoff_onset_median'].sel(longitude=slice(-80,-65),latitude=slice(-20,-60)).coarsen(latitude=10,longitude=10,boundary='trim').median().compute()
# # amco_coarsened_10_da
# # amco_coarsened_10_da.to_netcdf('amco_S_coarsened_10_da.nc')

# amco_N_coarsened_10_da = xr.open_dataarray('amco_N_coarsened_10_da.nc')
# amco_N_coarsened_10_da

# amco_S_coarsened_10_da = xr.open_dataarray('amco_S_coarsened_10_da.nc')
# amco_S_coarsened_10_da

In [ ]:
global_coarsened_100_ds = xr.open_dataset('global_coarsened_100_ds.nc') #.sel(longitude=slice(-100,100),latitude=slice(30,50))
global_coarsened_100_ds['runoff_onset_anomaly'] = global_coarsened_100_ds['runoff_onset'] - global_coarsened_100_ds['runoff_onset_median']
global_coarsened_100_ds = global_coarsened_100_ds.rio.write_crs('EPSG:4326')
global_coarsened_100_ds

In [ ]:
global_coarsened_100_ds['runoff_onset'].plot.imshow(col='water_year',col_wrap=5,robust=True)

In [ ]:
global_coarsened_100_ds['runoff_onset_anomaly'].sel(latitude=slice(70,30)).plot.imshow(col='water_year',col_wrap=5,robust=True,cmap='RdBu',figsize=(20,10))

In [ ]:
f,ax=plt.subplots(figsize=(20,7),subplot_kw=dict(projection=ccrs.Robinson()),dpi=300) #ccrs.Mollweide()
#global_coarsened_100_ds['runoff_onset_median'].plot(ax=ax,cmap='viridis',vmin=0,vmax=365,transform=ccrs.PlateCarree())
global_coarsened_100_ds['runoff_onset_median'].plot(ax=ax,cmap='viridis',vmin=110,vmax=250,transform=ccrs.PlateCarree(),cbar_kwargs={'label':'DOWY','orientation':'horizontal','shrink':0.3,'pad':0.04,'aspect':30})

#global_coarsened_100_ds['runoff_onset_mad'].plot(ax=ax,cmap='Reds',vmin=0,vmax=60,transform=ccrs.PlateCarree())
ax.coastlines()
# ax.set_xlim([-100,100])
# ax.set_ylim([30,50])
#ctx.add_basemap(ax, crs='EPSG:4326')
ax.set_title('2015-2024 median snowmelt runoff onset')

gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
gl.top_labels=False
gl.right_labels=False

f.tight_layout()


f.savefig('global_2014_2024_median_snowmelt_runoff_onset_map.png')

In [ ]:
f,ax=plt.subplots(figsize=(20,7),subplot_kw=dict(projection=ccrs.Robinson()),dpi=300) #ccrs.Mollweide()
#global_coarsened_100_ds['runoff_onset_median'].plot(ax=ax,cmap='viridis',vmin=0,vmax=365,transform=ccrs.PlateCarree())
#global_coarsened_100_ds['runoff_onset_median'].plot(ax=ax,cmap='viridis',robust=True,transform=ccrs.PlateCarree())

global_coarsened_100_ds['runoff_onset_mad'].plot(ax=ax,cmap='Reds',vmin=0,vmax=30,transform=ccrs.PlateCarree(),cbar_kwargs={'label':'days','orientation':'horizontal','shrink':0.3,'pad':0.04,'aspect':30})
ax.coastlines()
# ax.set_xlim([-100,100])
# ax.set_ylim([30,50])
#ctx.add_basemap(ax, crs='EPSG:4326')
ax.set_title('2015-2024 snowmelt runoff onset median absolute deviation')

gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
gl.top_labels=False
gl.right_labels=False

f.tight_layout()

f.savefig('global_2014_2024_snowmelt_runoff_onset_mad_map.png')

In [ ]:
f,ax=plt.subplots(figsize=(10,10),subplot_kw=dict(projection=ccrs.Robinson()),dpi=300) #ccrs.Mollweide()
#global_coarsened_100_ds['runoff_onset_median'].plot(ax=ax,cmap='viridis',vmin=0,vmax=365,transform=ccrs.PlateCarree())
#global_coarsened_100_ds['runoff_onset_median'].plot(ax=ax,cmap='viridis',robust=True,transform=ccrs.PlateCarree())

global_coarsened_100_ds['runoff_onset_median'].plot(ax=ax,cmap='viridis',robust=True,transform=ccrs.PlateCarree(),cbar_kwargs={'label':'DOWY'})
ax.coastlines()
# ax.set_xlim([-100,100])
# ax.set_ylim([30,50])
#ctx.add_basemap(ax, crs='EPSG:4326')
ax.set_title('2015-2024 median snowmelt runoff onset')



ax.set_extent([-135, -68, -60, 70], ccrs.PlateCarree())
gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
gl.top_labels=False
gl.right_labels=False

f.tight_layout()
f.savefig('american_cordillera_2014_2024_snowmelt_runoff_onset_map.png')



In [ ]:
# f,ax=plt.subplots(figsize=(8,10),subplot_kw=dict(projection=ccrs.AlbersEqualArea(central_latitude=50,central_longitude=-122.5)),dpi=300) #ccrs.Mollweide()
# #global_coarsened_100_ds['runoff_onset_median'].plot(ax=ax,cmap='viridis',vmin=0,vmax=365,transform=ccrs.PlateCarree())
# #global_coarsened_100_ds['runoff_onset_median'].plot(ax=ax,cmap='viridis',robust=True,transform=ccrs.PlateCarree())

# global_coarsened_100_ds['runoff_onset_median'].plot(ax=ax,cmap='viridis',robust=True,transform=ccrs.PlateCarree(),add_colorbar=False)
# ax.coastlines()
# # ax.set_xlim([-100,100])
# # ax.set_ylim([30,50])
# #ctx.add_basemap(ax, crs='EPSG:4326')
# ax.set_title('2015-2024 median snowmelt runoff onset')



# ax.set_extent([-140, -105, 30, 70], ccrs.PlateCarree())
# #ax.set_xlim([-140, -105])


# gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
# gl.top_labels=False
# gl.right_labels=False
# #ax.stock_img()
# #ax.set_xlim([-140, -105])

# f.tight_layout()
# f.savefig('american_cordillera_N_2014_2024_snowmelt_runoff_onset_map.png')

f,ax=plt.subplots(figsize=(8,10),subplot_kw=dict(projection=ccrs.AlbersEqualArea(central_latitude=50,central_longitude=-122.5)),dpi=300) #ccrs.Mollweide()
#global_coarsened_100_ds['runoff_onset_median'].plot(ax=ax,cmap='viridis',vmin=0,vmax=365,transform=ccrs.PlateCarree())
#global_coarsened_100_ds['runoff_onset_median'].plot(ax=ax,cmap='viridis',robust=True,transform=ccrs.PlateCarree())

amco_N_coarsened_10_da.plot(ax=ax,cmap='viridis',vmin=110,vmax=250,transform=ccrs.PlateCarree(),add_colorbar=False)
ax.coastlines()
# ax.set_xlim([-100,100])
# ax.set_ylim([30,50])
#ctx.add_basemap(ax, crs='EPSG:4326')
ax.set_title('2015-2024 median snowmelt runoff onset')



ax.set_extent([-150, -105, 30, 71], ccrs.PlateCarree())
#ax.set_xlim([-140, -105])


gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
gl.top_labels=False
gl.right_labels=False
#ax.stock_img()
#ax.set_xlim([-140, -105])

f.tight_layout()
f.savefig('american_cordillera_N_2014_2024_snowmelt_runoff_onset_map.png')

In [ ]:
# f,ax=plt.subplots(figsize=(5,8),subplot_kw=dict(projection=ccrs.AlbersEqualArea(central_latitude=-40,central_longitude=-72.5)),dpi=300) #ccrs.Mollweide()
# #global_coarsened_100_ds['runoff_onset_median'].plot(ax=ax,cmap='viridis',vmin=0,vmax=365,transform=ccrs.PlateCarree())
# #global_coarsened_100_ds['runoff_onset_median'].plot(ax=ax,cmap='viridis',robust=True,transform=ccrs.PlateCarree())

# global_coarsened_100_ds['runoff_onset_median'].plot(ax=ax,cmap='viridis',robust=True,transform=ccrs.PlateCarree(),add_colorbar=False)
# ax.coastlines()
# # ax.set_xlim([-100,100])
# # ax.set_ylim([30,50])
# #ctx.add_basemap(ax, crs='EPSG:4326')
# ax.set_title('2015-2024 median snowmelt runoff onset')



# ax.set_extent([-80, -65, -60, -20], ccrs.PlateCarree())
# gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
# gl.top_labels=False
# gl.right_labels=False

# f.tight_layout()
# f.savefig('american_cordillera_S_2014_2024_snowmelt_runoff_onset_map.png')

f,ax=plt.subplots(figsize=(5,8),subplot_kw=dict(projection=ccrs.AlbersEqualArea(central_latitude=-40,central_longitude=-67.5)),dpi=300) #ccrs.Mollweide()
#global_coarsened_100_ds['runoff_onset_median'].plot(ax=ax,cmap='viridis',vmin=0,vmax=365,transform=ccrs.PlateCarree())
#global_coarsened_100_ds['runoff_onset_median'].plot(ax=ax,cmap='viridis',robust=True,transform=ccrs.PlateCarree())

amco_S_coarsened_10_da.plot(ax=ax,cmap='viridis',vmin=110,vmax=250,transform=ccrs.PlateCarree(),add_colorbar=False)
ax.coastlines()
# ax.set_xlim([-100,100])
# ax.set_ylim([30,50])
#ctx.add_basemap(ax, crs='EPSG:4326')
ax.set_title('2015-2024 median snowmelt runoff onset')


import matplotlib.ticker as mticker
ax.set_extent([-75.1, -65, -60, -19.6], ccrs.PlateCarree())
gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
gl.xlocator = mticker.FixedLocator([-75, -70, -65])
gl.top_labels=False
gl.right_labels=False

f.tight_layout()
f.savefig('american_cordillera_S_2014_2024_snowmelt_runoff_onset_map.png')

In [ ]:
f,ax=plt.subplots(figsize=(5,10),subplot_kw=dict(projection=ccrs.Robinson()),dpi=300) #ccrs.Mollweide()
#global_coarsened_100_ds['runoff_onset_median'].plot(ax=ax,cmap='viridis',vmin=0,vmax=365,transform=ccrs.PlateCarree())
#global_coarsened_100_ds['runoff_onset_median'].plot(ax=ax,cmap='viridis',robust=True,transform=ccrs.PlateCarree())

global_coarsened_100_ds['runoff_onset_median'].plot(ax=ax,cmap='viridis',robust=True,transform=ccrs.PlateCarree(),cbar_kwargs={'label':'DOWY'})
ax.coastlines()
# ax.set_xlim([-100,100])
# ax.set_ylim([30,50])
#ctx.add_basemap(ax, crs='EPSG:4326')
ax.set_title('2015-2024 snowmelt runoff onset median')



ax.set_extent([-135, -105, -60, 70], ccrs.PlateCarree())
gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
gl.top_labels=False
gl.right_labels=False

f.tight_layout()
f.savefig('american_cordillera_S_2014_2024_snowmelt_runoff_onset_map.png')

In [ ]:
global_coarsened_100_ds['runoff_onset_median']

In [ ]:
f,ax=plt.subplots(figsize=(20,7),subplot_kw=dict(projection=ccrs.Robinson()),dpi=300) #ccrs.Mollweide()
#global_coarsened_100_ds['runoff_onset_median'].plot(ax=ax,cmap='viridis',vmin=0,vmax=365,transform=ccrs.PlateCarree())
global_coarsened_100_ds['runoff_onset_median'].sel(latitude=slice(90,20)).plot(ax=ax,cmap='viridis',robust=True,transform=ccrs.PlateCarree())

#global_coarsened_100_ds['runoff_onset_mad'].plot(ax=ax,cmap='Reds',vmin=0,vmax=60,transform=ccrs.PlateCarree())
ax.coastlines()
# ax.set_xlim([-100,100])
# ax.set_ylim([30,50])
#ctx.add_basemap(ax, crs='EPSG:4326')
ax.set_title('2015-2024 median snowmelt runoff onset')

f.tight_layout()


f.savefig('NH_2014_2024_median_snowmelt_runoff_onset_map.png')

In [ ]:
plot = global_ds['runoff_onset'].sel(latitude=slice(50,32),longitude=slice(-125,-105)).plot.imshow(col='water_year',col_wrap=10,robust=True,add_colorbar=False)
plot.fig.set_size_inches(100,10)
plot.fig.dpi = 300
plot.fig.suptitle('Western US 2014-2024 Snowmelt runoff onset (top) and anomaly (bottom)')
plot.fig.tight_layout()
plot.fig.savefig('WUS_2014_2024_snowmelt_runoff_onset_strip.png')

In [ ]:
plot = global_coarsened_100_ds['runoff_onset_anomaly'].sel(latitude=slice(50,32),longitude=slice(-125,-105)).plot.imshow(col='water_year',col_wrap=10,robust=True,add_colorbar=False)
plot.fig.set_size_inches(100,10)
plot.fig.dpi = 300
#plot.fig.suptitle('Western US 2014-2024 Snowmelt runoff onset (top) and anomaly (bottom)')
plot.fig.tight_layout()
plot.fig.savefig('WUS_2014_2024_snowmelt_runoff_onset_anomaly_strip.png')

In [ ]:
plot = global_coarsened_100_ds['runoff_onset'].sel(latitude=slice(67,63),longitude=slice(-25,-12)).plot.imshow(col='water_year',col_wrap=10,robust=True,add_colorbar=False)
plot.fig.set_size_inches(100,10)
plot.fig.dpi = 300
plot.fig.suptitle('Iceland 2014-2024 Snowmelt runoff onset (top) and anomaly (bottom)')
plot.fig.tight_layout()
plot.fig.savefig('Iceland_2014_2024_snowmelt_runoff_onset_strip.png')

In [ ]:
plot = global_coarsened_100_ds['runoff_onset_anomaly'].sel(latitude=slice(66.6,63),longitude=slice(-25,-12)).plot.imshow(col='water_year',col_wrap=10,robust=True,add_colorbar=False)
plot.fig.set_size_inches(100,10)
plot.fig.dpi = 300
#plot.fig.suptitle('Iceland 2014-2024 Snowmelt runoff onset (top) and anomaly (bottom)')
plot.fig.tight_layout()
plot.fig.savefig('Iceland_2014_2024_snowmelt_runoff_onset_anomaly_strip.png')

In [ ]:
iceland_coarsened_ds = global_ds.sel(latitude=slice(66.6,63),longitude=slice(-25,-13)).coarsen(latitude=2,longitude=2,boundary='trim').median().rio.reproject('EPSG:32627')
iceland_coarsened_ds['runoff_onset_anomaly'] = iceland_coarsened_ds['runoff_onset'] - iceland_coarsened_ds['runoff_onset_median']
iceland_coarsened_ds=iceland_coarsened_ds.rio.write_crs('EPSG:32627')
iceland_coarsened_ds
iceland_coarsened_ds.to_netcdf("iceland.nc")

In [ ]:
iceland_coarsened_ds = xr.open_dataset('iceland.nc')#.sel(x=slice(320000,870000),y=slice(7.4e6,7.01e6))
iceland_coarsened_ds

In [ ]:
# iceland_coarsened_ds = iceland_coarsened_ds.coarsen(x=2,y=2,boundary='trim').mean().compute()
# iceland_coarsened_ds 

In [ ]:
plot = iceland_coarsened_ds['runoff_onset'].plot(col='water_year',col_wrap=10,robust=True,add_colorbar=True,cbar_kwargs={'label':'DOWY'})
#plot.fig.set_size_inches(100,10)
plot.fig.dpi = 300
#plot.fig.suptitle('Iceland 2014-2024 Snowmelt runoff onset (top) and anomaly (bottom)')
#plot.fig.tight_layout()

for ax,wy in zip(plot.axs.flat,iceland_coarsened_ds.water_year.values):
    #ctx.add_basemap(ax,crs="EPSG:32627",attribution=False)
    ax.set_title(f'WY{wy}')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.axis('off')
    
plot.fig.savefig('Iceland_2014_2024_snowmelt_runoff_onset_strip_fullres_nobr.png')

In [ ]:
plot = iceland_coarsened_ds['runoff_onset_anomaly'].plot(col='water_year',col_wrap=10,robust=True,add_colorbar=True,cbar_kwargs={'label':'Days'},cmap='RdBu')
#plot.fig.set_size_inches(100,10)
plot.fig.dpi = 300
#plot.fig.suptitle('Iceland 2014-2024 Snowmelt runoff onset (top) and anomaly (bottom)')
#plot.fig.tight_layout()

for ax,wy in zip(plot.axs.flat,iceland_coarsened_ds.water_year.values):
    #ctx.add_basemap(ax,crs="EPSG:32627",attribution=False)
    ax.set_title(f'WY{wy}')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.axis('off')
    
plot.fig.savefig('Iceland_2014_2024_snowmelt_runoff_onset_anomaly_strip_fullres_nobr.png')

In [ ]:
global_coarsened_100_ds['runoff_onset_anomaly'].sel(latitude=slice(90,20)).plot.imshow(col='water_year',col_wrap=5,robust=True)

In [ ]:
# f, axs = plt.subplots(10, 2, figsize=(20, 40), 
#                       subplot_kw=dict(projection=ccrs.Mollweide()),
#                       dpi=300)

# for i, year in enumerate(global_coarsened_100_ds.water_year.values):
#     # Left panel - runoff onset
#     global_coarsened_100_ds['runoff_onset'].sel(water_year=year).plot(
#         ax=axs[i,0],
#         transform=ccrs.PlateCarree(),
#         vmin=0,
#         vmax=365,
#         cmap='viridis',
#         add_colorbar=True,
#         cbar_kwargs={'orientation': 'horizontal', 'shrink': 0.8, 'aspect': 40, 'label': 'Day of year'}
#     )
#     axs[i,0].coastlines()
#     axs[i,0].set_title(f'Runoff Onset {year}')
    
#     # Right panel - runoff onset anomaly
#     global_coarsened_100_ds['runoff_onset_anomaly'].sel(water_year=year).plot(
#         ax=axs[i,1],
#         transform=ccrs.PlateCarree(),
#         vmin=-60,
#         vmax=60,
#         cmap='RdBu_r',
#         add_colorbar=True,
#         cbar_kwargs={'orientation': 'horizontal', 'shrink': 0.8, 'aspect': 40, 'label': 'Days'}
#     )
#     axs[i,1].coastlines()
#     axs[i,1].set_title(f'Runoff Onset Anomaly {year}')

# plt.tight_layout()


In [ ]:
f, axs = plt.subplots(10, 2, figsize=(20, 30), 
                      subplot_kw=dict(projection=ccrs.Robinson()),
                      dpi=300)

for i, year in enumerate(global_coarsened_100_ds.water_year.values):
    # Left panel - runoff onset
    im1 = global_coarsened_100_ds['runoff_onset'].sel(
        water_year=year, 
        latitude=slice(90, 20)
    ).plot(
        ax=axs[i,0],
        transform=ccrs.PlateCarree(),
        vmin=110,
        vmax=250,
        cmap='viridis',
        add_colorbar=False
    )
    axs[i,0].coastlines()
    axs[i,0].set_extent([-180, 180, 20, 90], ccrs.PlateCarree())
    axs[i,0].set_title('')
    
    # Right panel - runoff onset anomaly  
    im2 = global_coarsened_100_ds['runoff_onset_anomaly'].sel(
        water_year=year,
        latitude=slice(90, 20)
    ).plot(
        ax=axs[i,1],
        transform=ccrs.PlateCarree(),
        vmin=-60,
        vmax=60,
        cmap='RdBu_r',
        add_colorbar=False
    )
    axs[i,1].coastlines()
    axs[i,1].set_extent([-180, 180, 20, 90], ccrs.PlateCarree())
    axs[i,1].set_title('')
axs[0,0].set_title('Runoff onset timing')
axs[0,1].set_title('Runoff onset anomaly')

# Add single colorbars at bottom
cbar_ax1 = f.add_axes([0.1, 0.05, 0.35, 0.02])
cbar_ax2 = f.add_axes([0.55, 0.05, 0.35, 0.02])
f.colorbar(im1, cax=cbar_ax1, orientation='horizontal', label='DOWY')
f.colorbar(im2, cax=cbar_ax2, orientation='horizontal', label='Days')

f.tight_layout()
#f.subplots_adjust(bottom=0.1)

f.savefig('NH_runoff_onset_and_anomalies_2015_2024.png', bbox_inches='tight', dpi=300)

In [ ]:
f, axs = plt.subplots(2, 10, figsize=(40, 8), 
                      subplot_kw=dict(projection=ccrs.Robinson()),
                      dpi=300)

for i, year in enumerate(global_coarsened_100_ds.water_year.values):
    # Top row - runoff onset
    im1 = global_coarsened_100_ds['runoff_onset'].sel(
        water_year=year, 
        latitude=slice(90, 20)
    ).plot(
        ax=axs[0,i],
        transform=ccrs.PlateCarree(),
        vmin=110,
        vmax=250,
        cmap='viridis',
        add_colorbar=False
    )
    axs[0,i].coastlines()
    axs[0,i].set_extent([-180, 180, 20, 90], ccrs.PlateCarree())
    
    # Bottom row - runoff onset anomaly
    im2 = global_coarsened_100_ds['runoff_onset_anomaly'].sel(
        water_year=year,
        latitude=slice(90, 20)
    ).plot(
        ax=axs[1,i],
        transform=ccrs.PlateCarree(),
        vmin=-60,
        vmax=60,
        cmap='RdBu_r',
        add_colorbar=False
    )
    axs[1,i].coastlines()
    axs[1,i].set_extent([-180, 180, 20, 90], ccrs.PlateCarree())

# Add colorbars on right side
cbar_ax1 = f.add_axes([0.92, 0.55, 0.02, 0.35])
cbar_ax2 = f.add_axes([0.92, 0.1, 0.02, 0.35])
f.colorbar(im1, cax=cbar_ax1, label='DOWY')
f.colorbar(im2, cax=cbar_ax2, label='Days')

plt.tight_layout()
f.subplots_adjust(right=0.9)

f.savefig('NH_runoff_onset_and_anomalies_2015_2024.png', bbox_inches='tight', dpi=300)

In [ ]:
f = global_coarsened_100_ds["runoff_onset"].plot(
    col="water_year",
    col_wrap=2,
    vmin=0,
    vmax=365,
    transform=ccrs.PlateCarree(),
    subplot_kws={
        "projection": ccrs.Mollweide(),
    },
    cbar_kwargs={"orientation": "horizontal", "shrink": 0.8, "aspect": 40},
)
f.map(lambda: plt.gca().coastlines())

In [ ]:
f = global_coarsened_100_ds["runoff_onset_anomaly"].plot(
    col="water_year",
    col_wrap=2,
    vmin=-60,
    vmax=60,
    cmap="RdBu_r",
    transform=ccrs.PlateCarree(),
    subplot_kws={
        "projection": ccrs.Mollweide(),
    },
    cbar_kwargs={"orientation": "horizontal", "shrink": 0.8, "aspect": 40},
)

f.map(lambda: plt.gca().coastlines())

In [ ]:
global_coarsened_100_ds['runoff_onset_median'].hvplot(cmap='viridis',clim=(0,365),width=1000, height=600)

In [ ]:
tile.geobox.boundingbox

In [ ]:
olympics_box = [-124.75,47.25,-122.75,48.5]
alps_box = [5.5,44,13.5,48]

hma_box = [5.5,27.5,13.5,48]


In [ ]:
tile = config.get_tile(20,38)

test_ds = global_ds.rio.clip_box(*alps_box,crs='EPSG:4326')
test_ds = test_ds.rio.reproject(test_ds.rio.estimate_utm_crs())
test_ds

In [ ]:
f,ax=plt.subplots(figsize=(10,10))
test_ds['runoff_onset_median'].plot(ax=ax,vmin=110,vmax=250)
ctx.add_basemap(ax=ax, crs=test_ds.rio.crs.to_string(),attribution=False) #source='Esri WorldImagery'
ax.set_aspect('equal')

In [ ]:
test_ds['runoff_onset_median'].hvplot(cmap='viridis',width=1000, height=600, tiles=True,project=True)

In [ ]:
global_coarsened_100_ds['runoff_onset_median'].hvplot.quadmesh(cmap='viridis',clim=(0,365),projection=ccrs.Orthographic(-90, 20), project=True,
    global_extent=True, coastline=True,width=1000, height=600)

In [ ]:
global_coarsened_100_ds['runoff_onset_mad'].hvplot(cmap='Reds',clim=(0,60))

In [ ]:
config.get_tile(20,37).geobox.explore(tiles=xyz.providers.Esri.WorldImagery)

In [ ]:
#view_tile(config.get_tile(27,39)) # v1

In [ ]:
view_tile(config.get_tile(65,75)) # v1

In [ ]:
view_tile(config.get_tile(10,109)) # v1

In [ ]:
view_tile(config.get_tile(29,41))

In [ ]:
view_tile(config.get_tile(23,39))

In [ ]:
view_tile(config.get_tile(23,129))

In [ ]:
view_tile(config.get_tile(9,11))

In [ ]:
test_ds = global_ds.rio.clip_box(-120,30,-110,50,crs='EPSG:4326')
test_ds

In [ ]:
f,axs=plt.subplots(1,2,figsize=(10,10))
test_ds['runoff_onset_median'].plot(ax=axs[0],vmin=0,vmax=365)
test_ds['runoff_onset_std'].plot(ax=axs[1],cmap='Reds')

for ax in axs:
    ctx.add_basemap(ax, crs=test_ds.rio.crs.to_string())

In [ ]:
test_ds['runoff_onset'].plot.imshow(col='water_year',col_wrap=3,vmin=0,vmax=365)

In [ ]:
import global_snowmelt_runoff_onset.processing as processing

tile = config.get_tile(14, 29)

s1_rtc_ds = processing.get_sentinel1_rtc(
    tile.geobox,
    config.bands,
    config.start_date,
    config.end_date,
    config.chunks_read,
)
s1_rtc_ds

s1_rtc_da = s1_rtc_ds['vv']
s1_rtc_WY_da = s1_rtc_da[s1_rtc_da['water_year']==2015]
s1_rtc_WY_da

In [ ]:
for relative_orbit in np.unique(s1_rtc_WY_da['sat:relative_orbit']):
    print(f'relative orbit: {relative_orbit}')
    s1_rtc_WY_da_relative_orbit = s1_rtc_WY_da[s1_rtc_WY_da['sat:relative_orbit'] == relative_orbit]
    print(f"{s1_rtc_WY_da_relative_orbit.time.sortby('time').diff('time').dt.days.values}")

In [ ]:
f,ax=plt.subplots(figsize=(20,5))
s1_rtc_WY_da['sat:relative_orbit'].plot.scatter(ax=ax,x='time')

In [ ]:
s1_rtc_WY_da[s1_rtc_WY_da['sat:relative_orbit']==79].isel(time=0).plot.imshow()


In [ ]:
s1_rtc_WY_da[s1_rtc_WY_da['sat:relative_orbit']==50].isel(time=0).plot.imshow()


In [ ]:
seasonal_snow_mask = xr.open_zarr(config.snow_phenology_store, consolidated=True, decode_coords='all') 
seasonal_snow_mask_clip_ds = seasonal_snow_mask.rio.clip_box(*tile.bbox_gdf.total_bounds,crs=tile.bbox_gdf.crs) # clip to correct box, maybe use total_bounds and then use crs 
seasonal_snow_mask_clip_ds

seasonal_snow_mask_clip_ds.to_dataarray().plot.imshow(row='water_year',col='variable',vmin=0,vmax=365)